# Cadence - Encoder Pipeline

## 1. Install dependencies and check notebook runtime

In [17]:
%pip install --upgrade torch torchao transformers litert-torch ai-edge-litert soundfile -q

In [18]:
import importlib.util
import shutil
import subprocess
from pathlib import Path

REQUIRED_PACKAGES = [
    "torch",
    "transformers",
    "litert_torch",
    "ai_edge_litert",
    "soundfile",
    "numpy",
]
missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]
if missing:
    raise RuntimeError(
        f"Missing required packages: {missing}. Run the install cell above, then restart the kernel if needed."
    )

for binary in ["ffmpeg", "ffprobe"]:
    if not shutil.which(binary):
        raise RuntimeError(
            f"{binary} was not found on PATH. Install FFmpeg and restart the notebook kernel."
        )

RECORDINGS_DIR = Path("/content/drive/MyDrive/Class Files/A2 CS 2025/Recordings")
WORK_DIR = Path("/content/cadence_encoder_work")
WAV_OUTPUT_DIR = WORK_DIR / "sample_wavs"
OUTPUT_DIR = Path("/content/cadence_outputs")

OUTPUT_MODEL_PATH = OUTPUT_DIR / "encoder_wav2vec2.tflite"
QUANT_MODEL_PATH = OUTPUT_DIR / 'encoder_wav2vec2_int8.tflite'

WORK_DIR.mkdir(parents=True, exist_ok=True)
WAV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Recordings dir: {RECORDINGS_DIR}")
print(f"Working files: {WORK_DIR}")
print(f"Final model: {OUTPUT_MODEL_PATH}")

if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(
        result.stdout
        if result.returncode == 0
        else "nvidia-smi is available but did not report a GPU."
    )
else:
    print("nvidia-smi not found. PyTorch CUDA availability is checked later.")

Recordings dir: /content/drive/MyDrive/Class Files/A2 CS 2025/Recordings
Working files: /content/cadence_encoder_work
Final model: /content/cadence_outputs/encoder_wav2vec2.tflite
Tue Apr 21 13:11:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             32W /   70W |     959M

## 2. Mount Google Drive and audit recordings

In [19]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
if not RECORDINGS_DIR.exists():
    raise FileNotFoundError(
        f"Recordings directory not found: {RECORDINGS_DIR}. "
        "Check that Drive is mounted and the folder path is correct."
    )

In [21]:
import json
import subprocess


def get_mp4_info(filepath):
    cmd = [
        "ffprobe",
        "-v",
        "quiet",
        "-print_format",
        "json",
        "-show_streams",
        "-show_format",
        str(filepath),
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode != 0:
            return {"error": result.stderr[-500:], "ok": False}

        data = json.loads(result.stdout)
        duration = float(data["format"].get("duration", 0))
        size_mb = filepath.stat().st_size / (1024 * 1024)

        sample_rate = None
        channels = None
        for stream in data.get("streams", []):
            if stream.get("codec_type") == "audio":
                sample_rate = int(stream.get("sample_rate", 0))
                channels = int(stream.get("channels", 0))
                break

        return {
            "duration_min": round(duration / 60, 1),
            "size_mb": round(size_mb, 1),
            "sample_rate": sample_rate,
            "channels": channels,
            "ok": duration > 0,
        }

    except Exception as e:
        return {"error": str(e), "ok": False}


mp4_files = sorted(RECORDINGS_DIR.glob("*.mp4"))
print(f"Found {len(mp4_files)} MP4 files in {RECORDINGS_DIR}.\n")

if not mp4_files:
    raise RuntimeError("No MP4 recordings found. Add recordings before continuing.")

total_duration_min = 0.0
total_size_mb = 0.0
corrupted = []

for i, fpath in enumerate(mp4_files):
    info = get_mp4_info(fpath)

    if not info["ok"]:
        corrupted.append(fpath.name)
        print(
            f"File {i + 1}: Filename: {fpath.name}, Status: ERROR, Detail: {info.get('error', 'unknown')}"
        )
        continue

    total_duration_min += info["duration_min"]
    total_size_mb += info["size_mb"]

    print(
        f"File {i + 1}: Filename: {fpath.name}, "
        f"Duration: {info['duration_min']}min, "
        f"Size: {info['size_mb']}MB, "
        f"SampleRate: {info['sample_rate']}Hz, "
        f"Channels: {info['channels']}"
    )

print(
    f"Total: {len(mp4_files)} files, {round(total_duration_min, 1)} minutes, {round(total_size_mb, 1)} MB."
)

if corrupted:
    print(f"\nUnreadable files: {len(corrupted)}")
    for filename in corrupted:
        print(f"- {filename}")
else:
    print("\nAll files readable.")

Found 81 MP4 files in /content/drive/MyDrive/Class Files/A2 CS 2025/Recordings.

File 1: Filename: 10_26-07-2025_Practical_P.3 - Part 1 - Airport Distances (Cont. 2) and Employee Salary Calculation + Validation.mp4, Duration: 73.3min, Size: 128.9MB, SampleRate: 48000Hz, Channels: 2
File 2: Filename: 12_02-08-2025_Theory_13.2 - Part 2 - Sequential and Direct File Access.mp4, Duration: 141.5min, Size: 295.6MB, SampleRate: 48000Hz, Channels: 2
File 3: Filename: 13_07-08-2025_Theory_13.2 - Part 3 - Hashing Algorithms.mp4, Duration: 100.9min, Size: 177.3MB, SampleRate: 48000Hz, Channels: 2
File 4: Filename: 14_09-08-2025_Practical_P.3 - Part 2 - WHILE & FOR Loops.mp4, Duration: 119.5min, Size: 196.0MB, SampleRate: 48000Hz, Channels: 2
File 5: Filename: 15_14-08-2025_Theory_13.3 - Part 1 - Fixed Point Notation.mp4, Duration: 118.8min, Size: 191.7MB, SampleRate: 48000Hz, Channels: 2
File 6: Filename: 16_16-08-2025_Practical_P.3 - Part 3 - Christmas Tree Coding Problem.mp4, Duration: 117.2min,

## 3. Convert 5 sample recordings to 16kHz mono WAV

In [22]:
SAMPLE_SIZE = 5

sample_files = [f for f in mp4_files if get_mp4_info(f)["ok"]][:SAMPLE_SIZE]
converted_wavs = []

for fpath in sample_files:
    out_path = WAV_OUTPUT_DIR / f"{fpath.stem}.wav"

    cmd = [
        "ffmpeg",
        "-i",
        str(fpath),
        "-ar",
        "16000",
        "-ac",
        "1",
        "-sample_fmt",
        "s16",
        "-y",
        str(out_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0 and out_path.exists():
        size_mb = out_path.stat().st_size / (1024 * 1024)
        print(f"OK {out_path.name} ({round(size_mb, 1)} MB)")
        converted_wavs.append(out_path)
    else:
        print(f"FAIL {fpath.name}")
        print(result.stderr[-500:])

print(
    f"\nConverted {len(converted_wavs)}/{len(sample_files)} files to {WAV_OUTPUT_DIR}."
)

if len(converted_wavs) != SAMPLE_SIZE:
    raise RuntimeError(
        "Expected 5 converted WAV files. Fix input recordings or FFmpeg before continuing."
    )

OK 10_26-07-2025_Practical_P.3 - Part 1 - Airport Distances (Cont. 2) and Employee Salary Calculation + Validation.wav (134.2 MB)
OK 12_02-08-2025_Theory_13.2 - Part 2 - Sequential and Direct File Access.wav (259.1 MB)
OK 13_07-08-2025_Theory_13.2 - Part 3 - Hashing Algorithms.wav (184.7 MB)
OK 14_09-08-2025_Practical_P.3 - Part 2 - WHILE & FOR Loops.wav (218.9 MB)
OK 15_14-08-2025_Theory_13.3 - Part 1 - Fixed Point Notation.wav (217.5 MB)

Converted 5/5 files to /content/cadence_encoder_work/sample_wavs.


## 4. Load and freeze Wav2Vec2 encoder

In [23]:
import torch
from transformers import Wav2Vec2Model

MODEL_NAME = "facebook/wav2vec2-base-960h"
print(f"Loading {MODEL_NAME}.")

encoder = Wav2Vec2Model.from_pretrained(MODEL_NAME)
encoder.eval()

for param in encoder.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in encoder.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Model size (float32): ~{total_params * 4 / 1024 / 1024:.1f} MB")
print(f"Model size (int8 estimate): ~{total_params / 1024 / 1024:.1f} MB")
print("Encoder loaded and frozen.")

Loading facebook/wav2vec2-base-960h.


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 94,371,712
Model size (float32): ~360.0 MB
Model size (int8 estimate): ~90.0 MB
Encoder loaded and frozen.


In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

encoder = encoder.to(device).eval()

Using device: cuda
CUDA device: Tesla T4


## 5. Inspect embeddings on sample classroom audio

In [25]:
import numpy as np
import soundfile as sf

CHUNK_SAMPLES = 160_000


def load_wav_chunk(wav_path, chunk_index=0):
    audio, sr = sf.read(str(wav_path), dtype="float32", always_2d=False)
    assert sr == 16000, f"Expected 16kHz, got {sr}Hz in {wav_path}"

    if audio.ndim > 1:
        audio = np.mean(audio, axis=1).astype(np.float32)

    start = chunk_index * CHUNK_SAMPLES
    end = start + CHUNK_SAMPLES
    chunk = audio[start:end]

    if len(chunk) < CHUNK_SAMPLES:
        chunk = np.pad(chunk, (0, CHUNK_SAMPLES - len(chunk)))

    return chunk.astype(np.float32)


def analyse_embeddings(embeddings_np):
    temporal_variance = float(np.mean(np.std(embeddings_np, axis=0)))

    norms = np.linalg.norm(embeddings_np, axis=1, keepdims=True)
    normalised = embeddings_np / (norms + 1e-8)
    frame_sims = np.sum(normalised[:-1] * normalised[1:], axis=1)

    return {
        "shape": embeddings_np.shape,
        "temporal_variance": round(temporal_variance, 4),
        "mean_adjacent_cosine": round(float(np.mean(frame_sims)), 4),
        "global_mean": round(float(np.mean(embeddings_np)), 4),
        "global_std": round(float(np.std(embeddings_np)), 4),
        "has_nan": bool(np.any(np.isnan(embeddings_np))),
        "has_inf": bool(np.any(np.isinf(embeddings_np))),
    }

In [26]:
print("Running encoder on sample classroom audio chunks.")
all_stats = []

for wav_path in converted_wavs:
    chunk = load_wav_chunk(wav_path, chunk_index=0)
    input_tensor = torch.from_numpy(chunk).unsqueeze(0).to(device)

    with torch.no_grad():
        output = encoder(input_values=input_tensor)
        embeddings_np = output.last_hidden_state.squeeze(0).cpu().numpy()

    stats = analyse_embeddings(embeddings_np)
    all_stats.append(stats)

    print(
        f"Filename: {wav_path.name}, "
        f"Shape: {stats['shape']}, "
        f"Variance: {stats['temporal_variance']}, "
        f"CosSim: {stats['mean_adjacent_cosine']}, "
        f"NaN: {'YES' if stats['has_nan'] else 'no'}, "
        f"Inf: {'YES' if stats['has_inf'] else 'no'}"
    )

if any(s["has_nan"] or s["has_inf"] for s in all_stats):
    raise RuntimeError(
        "NaN or Inf detected in encoder embeddings. Check audio conversion."
    )

mean_variance = float(np.mean([s["temporal_variance"] for s in all_stats]))
print(f"Mean temporal variance across {len(all_stats)} files: {mean_variance:.4f}.")
print("All embeddings clean. Frozen encoder verified on sample audio.")

Running encoder on sample classroom audio chunks.
Filename: 10_26-07-2025_Practical_P.3 - Part 1 - Airport Distances (Cont. 2) and Employee Salary Calculation + Validation.wav, Shape: (499, 768), Variance: 0.0014, CosSim: 1.0, NaN: no, Inf: no
Filename: 12_02-08-2025_Theory_13.2 - Part 2 - Sequential and Direct File Access.wav, Shape: (499, 768), Variance: 0.0003, CosSim: 1.0, NaN: no, Inf: no
Filename: 13_07-08-2025_Theory_13.2 - Part 3 - Hashing Algorithms.wav, Shape: (499, 768), Variance: 0.0003, CosSim: 1.0, NaN: no, Inf: no
Filename: 14_09-08-2025_Practical_P.3 - Part 2 - WHILE & FOR Loops.wav, Shape: (499, 768), Variance: 0.0009, CosSim: 1.0, NaN: no, Inf: no
Filename: 15_14-08-2025_Theory_13.3 - Part 1 - Fixed Point Notation.wav, Shape: (499, 768), Variance: 0.1432, CosSim: 0.6978, NaN: no, Inf: no
Mean temporal variance across 5 files: 0.0292.
All embeddings clean. Frozen encoder verified on sample audio.


## 6. Wrap encoder for fixed-shape export

In [27]:
import torch.nn as nn
import torch.nn.functional as F

TARGET_FRAMES = 500
EMBEDDING_DIM = 768


class Wav2Vec2EncoderWrapper(nn.Module):
    def __init__(self, wav2vec2_model):
        super().__init__()
        self.encoder = wav2vec2_model

    def forward(self, x):
        # Input: float32[1, 160000]. Output: float32[1, 500, 768].
        output = self.encoder(input_values=x)
        embeddings = output.last_hidden_state

        time_frames = embeddings.shape[1]
        if time_frames < TARGET_FRAMES:
            embeddings = F.pad(embeddings, (0, 0, 0, TARGET_FRAMES - time_frames))
        elif time_frames > TARGET_FRAMES:
            embeddings = embeddings[:, :TARGET_FRAMES, :]

        return embeddings

In [28]:
wrapped_model = Wav2Vec2EncoderWrapper(encoder).eval()

dummy_input = torch.zeros(1, CHUNK_SAMPLES, dtype=torch.float32).to(device)
with torch.no_grad():
    test_output = wrapped_model(dummy_input)

print(f"Input shape: {tuple(dummy_input.shape)}")
print(f"Output shape: {tuple(test_output.shape)}")

assert tuple(test_output.shape) == (1, TARGET_FRAMES, EMBEDDING_DIM), (
    f"Shape mismatch: expected (1, {TARGET_FRAMES}, {EMBEDDING_DIM}), got {tuple(test_output.shape)}."
)
print("Wrapper shape contract verified.")

Input shape: (1, 160000)
Output shape: (1, 500, 768)
Wrapper shape contract verified.


## 7. INT8 quantisation + TFLite export

In [29]:
import torchao
import litert_torch

print("torchao:", torchao.__version__)
print("litert_torch import OK")

torchao: 0.17.0
litert_torch import OK


In [31]:
wrapped_model_cpu = wrapped_model.cpu().eval()

sample_input = (torch.zeros(1, CHUNK_SAMPLES, dtype=torch.float32),)

print("Converting PyTorch to LiteRT/TFLite via litert_torch.")

edge_model = litert_torch.convert(wrapped_model_cpu, sample_input)
edge_model.export(str(OUTPUT_MODEL_PATH))

size_mb = OUTPUT_MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"\nTFLite saved: {OUTPUT_MODEL_PATH} ({size_mb:.1f} MB)")

Converting PyTorch to LiteRT/TFLite via litert_torch.

TFLite saved: /content/cadence_outputs/encoder_wav2vec2.tflite (360.5 MB)


In [32]:
import tensorflow as tf

_ai_edge_converter_flags = {"optimizations": [tf.lite.Optimize.DEFAULT]}

print("Quantizing with built-in dynamic range quantisation.")

edge_model_int8 = litert_torch.convert(
    wrapped_model_cpu, sample_input, _ai_edge_converter_flags=_ai_edge_converter_flags
)
edge_model_int8.export(str(QUANT_MODEL_PATH))

size_int8_mb = QUANT_MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"\nQuantised TFLite saved: {QUANT_MODEL_PATH} ({size_int8_mb:.1f} MB)")

Quantizing with built-in dynamic range quantisation...

Quantised TFLite saved: /content/cadence_outputs/encoder_wav2vec2_int8.tflite (91.9 MB)


## 8. Verify the INT8 TFLite model

In [33]:
import numpy as np
from ai_edge_litert.interpreter import Interpreter

interpreter = Interpreter(model_path=str(QUANT_MODEL_PATH))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

checks = {
    "input_name": input_details[0]["name"],
    "input_shape": tuple(input_details[0]["shape"].tolist()),
    "input_dtype": input_details[0]["dtype"],
    "output_name": output_details[0]["name"],
    "output_shape": tuple(output_details[0]["shape"].tolist()),
    "output_dtype": output_details[0]["dtype"],
}

print("TFLite Tensor Contract Verification")
print(f"Input name: {checks['input_name']}")
print(f"Input shape: {checks['input_shape']}")
print(f"Input dtype: {checks['input_dtype']}")
print(f"Output name: {checks['output_name']}")
print(f"Output shape: {checks['output_shape']}")
print(f"Output dtype: {checks['output_dtype']}")

zero_input = np.zeros((1, CHUNK_SAMPLES), dtype=np.float32)
interpreter.set_tensor(input_details[0]["index"], zero_input)
interpreter.invoke()
zero_output = interpreter.get_tensor(output_details[0]["index"])

real_chunk = load_wav_chunk(converted_wavs[0], chunk_index=0).reshape(1, -1)
interpreter.set_tensor(input_details[0]["index"], real_chunk)
interpreter.invoke()
real_output = interpreter.get_tensor(output_details[0]["index"])
real_variance = float(np.mean(np.std(real_output.squeeze(0), axis=0)))

verification_results = [
    ("Input shape == (1, 160000)", checks["input_shape"] == (1, CHUNK_SAMPLES)),
    ("Input dtype == float32", checks["input_dtype"] == np.float32),
    (
        "Output shape == (1, 500, 768)",
        checks["output_shape"] == (1, TARGET_FRAMES, EMBEDDING_DIM),
    ),
    ("Output dtype == float32", checks["output_dtype"] == np.float32),
    ("No NaN in zero-input output", not bool(np.any(np.isnan(zero_output)))),
    ("No Inf in zero-input output", not bool(np.any(np.isinf(zero_output)))),
    ("No NaN in real-audio output", not bool(np.any(np.isnan(real_output)))),
    ("No Inf in real-audio output", not bool(np.any(np.isinf(real_output)))),
]
name_results = [
    ("Input name == input_values", checks["input_name"] == "input_values"),
    ("Output name == embeddings", checks["output_name"] == "embeddings"),
]

all_passed = True
names_passed = True

print("\nVerification Results")
for label, passed in verification_results:
    print(f"- [{'PASS' if passed else 'FAIL'}] {label}")
    all_passed = all_passed and passed

for label, passed in name_results:
    print(f"- [{'PASS' if passed else 'INFO'}] {label}")
    names_passed = names_passed and passed

print(f"Real-audio embedding variance: {real_variance:.4f}")

if not all_passed:
    raise RuntimeError("TFLite verification failed.")

if not names_passed:
    print("Tensor names differ from the original contract.")

print("TFLite verification passed.")

TFLite Tensor Contract Verification
Input name: serving_default_args_0:0
Input shape: (1, 160000)
Input dtype: <class 'numpy.float32'>
Output name: StatefulPartitionedCall:0
Output shape: (1, 500, 768)
Output dtype: <class 'numpy.float32'>

Verification Results
- [PASS] Input shape == (1, 160000)
- [PASS] Input dtype == float32
- [PASS] Output shape == (1, 500, 768)
- [PASS] Output dtype == float32
- [PASS] No NaN in zero-input output
- [PASS] No Inf in zero-input output
- [PASS] No NaN in real-audio output
- [PASS] No Inf in real-audio output
- [INFO] Input name == input_values
- [INFO] Output name == embeddings
Real-audio embedding variance: 0.0037
Tensor names differ from the original contract.
TFLite verification passed.
